## This notebook will walk you through the process of changing the enformer class values again (resampling of random class from enformer_high and enformer_low based on their absolute number)

In [3]:
import pandas as pd 
import numpy as np
from ast import literal_eval
import yaml
from importlib import reload

# load helpful functions
import sys
sys.path.append('../../00_helpful_functions')
import helpful_functions as hf
reload(hf)

# config 
config_path = "/home/kisa/coding/80K_MPRA/80K-Analysis/global80K_config.yaml"
with open(config_path) as conf:
    config = yaml.load(conf, Loader=yaml.FullLoader)
    conf.close()

In [2]:
# read the enformer file
common_enformer_varaint_df = pd.read_csv(config['files']['creating']['common_and_enformer_variants'], sep="\t")
common_enformer_varaint_df['enformer_variant_info'] = common_enformer_varaint_df['enformer_variant_info'].apply(literal_eval)
# common_enformer_varaint_df['enformer_class_list'] = common_enformer_varaint_df['enformer_variant_info'].apply(hf.get_enformer_class_list)
# common_enformer_varaint_df['gene_set_list'] = common_enformer_varaint_df['enformer_variant_info'].apply(hf.get_gene_set_list)
# common_enformer_varaint_df['variant_type_list'] = common_enformer_varaint_df['enformer_variant_info'].apply(hf.get_variant_type_list)

enformer_class_df = common_enformer_varaint_df.loc[~common_enformer_varaint_df['DNase_max'].isna()]

# Explode the list of tuples into separate rows
df_exploded = enformer_class_df.explode('enformer_variant_info')

# Split the tuples into separate columns
df_exploded[['gene_set', 'variant_type', 'enformer_class']] = pd.DataFrame(df_exploded['enformer_variant_info'].tolist(), index=df_exploded.index)

##### Check for duplicates in common_and_prioritized variant file

In [3]:
duplicates_enformer_info = common_enformer_varaint_df.loc[common_enformer_varaint_df.duplicated(subset=['chr_pos_ref_alt'], keep=False)].sort_values(by="POS")
duplicates_enformer_info

,CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,chr_pos_ref_alt,DNase_max,max_col,enformer_variant_info


##### Investigate Numbers of variants

In [4]:
print('Number of variants from exploded df: (35000 expected) ', df_exploded.shape[0])
df_exploded.head()

Number of variants from exploded df: (35000 expected)  35000


,CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,chr_pos_ref_alt,DNase_max,max_col,enformer_variant_info,gene_set,variant_type,enformer_class
34122,chr6,43797164,VEGFA|ENSG00000112715.26|EH38E3709352|6-437971...,G,A,1,PASS,AF=6.57168e-06;AC=1,6-43797164-G-A,4254.3410,392_DNASE:CD14-positive,"(cardiac, singleton, enformer_high)",cardiac,singleton,enformer_high
34123,chr18,3059465,MYOM1|ENSG00000101605.14|EH38E3253954|18-30594...,T,C,1,PASS,AF=6.56866e-06;AC=1,18-3059465-T-C,4022.5996,"177_DNASE:CD8-positive,","(cardiac, singleton, enformer_high)",cardiac,singleton,enformer_high
34124,chr6,43797016,VEGFA|ENSG00000112715.26|EH38E3709352|6-437970...,G,A,1,PASS,AF=6.57618e-06;AC=1,6-43797016-G-A,3591.8310,392_DNASE:CD14-positive,"(cardiac, singleton, enformer_high)",cardiac,singleton,enformer_high
34125,chr12,2616967,CACNA1C|ENSG00000151067.23|EH38E2998634|12-261...,C,A,1,PASS,AF=6.56763e-06;AC=1,12-2616967-C-A,2754.7705,401_DNASE:common myeloid,"(cardiac, singleton, enformer_high)",cardiac,singleton,enformer_high
34125,chr12,2616967,CACNA1C|ENSG00000151067.23|EH38E2998634|12-261...,C,A,1,PASS,AF=6.56763e-06;AC=1,12-2616967-C-A,2754.7705,401_DNASE:common myeloid,"(neuro, singleton, enformer_high)",neuro,singleton,enformer_high


In [5]:
# investigate the numbers of the enformer_classes
df_exploded.groupby(['gene_set', 'variant_type'])['enformer_class'].value_counts()

gene_set  variant_type  enformer_class 
cardiac   singleton     enformer_high      3500
                        enformer_low        750
                        enformer_random     750
          ultra-rare    enformer_high      3500
                        enformer_low        750
                        enformer_random     750
cava      singleton     enformer_high      3500
                        enformer_low        750
                        enformer_random     750
          ultra-rare    enformer_high      3500
                        enformer_low        750
                        enformer_random     750
neuro     singleton     enformer_high      3500
                        enformer_low        750
                        enformer_random     750
          ultra-rare    enformer_high      3500
                        enformer_low        750
                        enformer_random     750
random    singleton     enformer_high      1750
                        enformer_low        375


In [6]:
# # example for one group (random, ultra-rare)
# enformer_high_percentage = 0.7
# enformer_low_percentage = 0.15
# for gene_set in ['random', 'cava', 'neuro', 'cardiac']:
#     print(gene_set)
#     chosen_variants = prioritized_variant_number[gene_set][0][1]
#     total_number = total_number_dict[gene_set][0][1]

#     number_enformer_high = enformer_high_percentage * chosen_variants
#     number_enformer_low = enformer_low_percentage * chosen_variants
#     # the following could be written as enformer_low_percentage / enformer_high_percentage (if I understood this correctly)
#     resampling_percentage = ((enformer_low_percentage * chosen_variants) / total_number) / ((enformer_high_percentage * chosen_variants) / total_number)

#     # get number of enformer high and enformer low which needs to be sampled to be random_effect
#     resample_number_high = number_enformer_high * resampling_percentage
#     resample_number_low = number_enformer_low * resampling_percentage

# # make a subset of the enformer file only looking on the 'random' and 'ultra-rare' variants

##### "Kopfrechnen"

In [7]:
0.15 / 0.7

0.2142857142857143

In [8]:
1750 * 0.21
375 * 0.21

78.75

In [9]:
(5000 * 0.7) / 598537

0.005847591711122286

### Start resampling

In [10]:
total_number_dict = {'cardiac': [('ultra-rare',289724), ('singleton', 296173)],
                     'cava': [('ultra-rare',69363), ('singleton', 69286)],
                     'neuro': [('ultra-rare',598537), ('singleton', 607227)],
                     'random': [('ultra-rare',23216), ('singleton', 24397)],
                    } # from Y1_MPRA_design.pptx

prioritized_variant_number = {'cardiac': [('ultra-rare',5000), ('singleton', 5000)],
                     'cava': [('ultra-rare',5000), ('singleton', 5000)],
                     'neuro': [('ultra-rare',5000), ('singleton', 5000)],
                     'random': [('ultra-rare',2500), ('singleton', 2500)],
                    }

3500
750
750

3500/290k = Anteil High
750/290k = Anteil Random
750/290k = Anteil Low

750 / (290k - 4250) = tatsächlicher Anteil Random

(290k - 4250) / 290k = 0.985
1 - 0.985 = 1.5% Samples aus Random entfernen = x
3500/(3500+750) = 82.35% * x aus High
(1 - 82.35%) * x aus Low

5000 

3500
750
750

69k
(69k - 4250) / 69k = 0.9384
1 - 0.9384 = 6.16% Samples aus Random entfernen = x
3500/(3500+750) = 82.35% * x aus High
(1 - 82.35%) * x aus Low

In [11]:
for all_variants in [69363, 289724, 598537]:
    sampling_ration_random = 750/(all_variants - 750 - 3500)
    # sampling_ration_random = 750/(all_variants - 750 - 3500)
    # sampling_ration_random = 750/(all_variants - 750 - 3500)
    print(sampling_ration_random)
    random_sampling_num = round(sampling_ration_random * 4250)
    high_sampling_num = round(sampling_ration_random * 3500)
    low_sampling_num = random_sampling_num - high_sampling_num
    print(random_sampling_num, high_sampling_num, low_sampling_num)
    print(750 + high_sampling_num + low_sampling_num)
    print((750 + high_sampling_num + low_sampling_num)/all_variants)
    # 3500 * sampling_ration_random

0.011518437178443629
49 40 9
799
0.011519109611752664
0.0026272094831753506
11 9 2
761
0.0026266377655976034
0.0012620165004450712
5 4 1
755
0.0012614090691135218


OK, also adjustierter Plan:
(1) Für high und low Anzahl zusampelnder Varianten bestimmen (mit dem sampling ratio) 
(2) Sampling ratio * (#LOW + #High) aus random entfernen und mit (sampling ratio von HIGH und LOW auffüllen)

Die gesampleten Varianten aus LOW und HIGH werden aber nicht aus diesen Gruppen rausgenommen, oder?

Und mit dem letzten Teil, dass die Anfangs und End-liste reduziert wird meinst du, dass wir elemente aus diesen listen nicht entfernen, sondern zusätzlich in unsere random gruppe aufnehmen. 

##### Procedure:
1. I computed for each group (e.g. 'neuro' and 'ultra-rare') the proportion of variation covered within the high and low enformer class
2. Based on this I computed the number of variants from each (enformer high and enformer low) I need to resample respectively 
3. In order to resample but stil have the proportion of high: 0.7, random: 0.15, low: 0.15 I need to remove this proportion of elements from the random set
4. Resampling from enformer high and enformer low and concatinating these informations to the dataframe itself

Ziel ist die Anzahl der varianten zu bestimmen, die wir von high und low in random packen, dabei wird jede Gruppe ('neuro', 'ultra-rare' und 'cardiac', 'singleton') unabhängig voneinander angeschaut, weil sich das Verhältnis der von enformer_high und enformer_low abgedeckten Variation unterscheided.

1. Genau dieses Verhältnis wurde berechnet: Der Anteil von Varianten aus der jeweiligen Gesamtanzahl, die von enformer high und enformer low abgedeckt werden.
2. Weil wir uns für das random set interessieren und trotz sampling das grundlegende Verhältnis von high: 0.7, random: 0.15, low: 0.15 beibehalten wollen haben wir nun die Anzahl der Random set varianten berechnet, die wir durch sampling von high und low ersetzten wollen.
3. Im Endeffekt haben wir dann aus dem Verhältnis von enformer high zu enformer low und der Anzahl der Varianten in random, die wir durch das sampling von high und low ersetzen müssen die Anzahl für jede Gruppe berechnet.
4. Nun wurde erstmal die Anzahl, die wir in der random class an platz brauchen gesampled und entfernt und dann jeweils von high und low gesampled, diese wurden dann an den dataset erneut concatenated mit enformer class auf random.

In [12]:
# Define the enformer percentages
enformer_high_percentage = 0.7
enformer_low_percentage = 0.15
enformer_random_percentage = 1 - (enformer_high_percentage + enformer_low_percentage)

# Function to calculate the resample numbers for each pair
def calculate_resample_numbers(gene_set, variant_type):
    chosen_variants = dict(prioritized_variant_number[gene_set])[variant_type]
    total_number = dict(total_number_dict[gene_set])[variant_type]
    # print(gene_set, variant_type)
    # print(chosen_variants)
    # print(total_number)
    chosen_variants = prioritized_variant_number[gene_set][0][1]
    total_number = total_number_dict[gene_set][0][1]

    number_enformer_high = enformer_high_percentage * chosen_variants
    number_enformer_low = enformer_low_percentage * chosen_variants
    number_enformer_random = chosen_variants - (number_enformer_high + number_enformer_low)
    
    # 750 / (290k - 4250) = tatsächlicher Anteil Random
    # Fraction of randoms among the remaining variant sets
    fraction_of_random = enformer_random_percentage * chosen_variants / (total_number - (number_enformer_high + number_enformer_low))

    # remove this fraction of enformer_random and fill with sampling of enformer high and low class values
    # meant to be the proportion of the random set to be available for the high and low classes
    # 1 - ((290k - 4250) / 290k) = 1.5% Samples aus Random entfernen = x
    # 1 - proportion of all variants available for the random set
    # the proportion of variants not available for the random set * number of enformer random => Number of variants in the current random set which need to be resampled from enformer high or low
    random_removable_rows = round((1 - ((total_number - (number_enformer_high + number_enformer_low)) / total_number)) * number_enformer_random)

    # 3500/(3500+750) = 82.35% * x aus High
    # (1 - 82.35%) * x aus Low
    # Number of samples required from high and low set repectively is the proportion of variants within the high or low set with the proportion of high and low respectively
    high_sample_percentage = number_enformer_high / (number_enformer_high + number_enformer_low)
    low_sample_percentage = 1 - high_sample_percentage
    resample_number_high = round(high_sample_percentage * random_removable_rows)
    resample_number_low = random_removable_rows - resample_number_high
    
    return int(resample_number_high), int(resample_number_low), random_removable_rows

# Apply the function to all gene_set and variant_type combinations
resampling_results = {}
for gene_set in total_number_dict.keys():
    resampling_results[gene_set] = {}
    for variant_type, _ in total_number_dict[gene_set]:
        resample_number_high, resample_number_low, random_removable_rows = calculate_resample_numbers(gene_set, variant_type)
        resampling_results[gene_set][variant_type] = {
            'resample_number_high': resample_number_high,
            'resample_number_low': resample_number_low,
            'random_removable_rows': random_removable_rows
        }

# Print the results
for gene_set, variant_types in resampling_results.items():
    for variant_type, numbers in variant_types.items():
        print(f"{gene_set} - {variant_type}: High - {numbers['resample_number_high']}, Low - {numbers['resample_number_low']}; removed_rows_from_random: {numbers['random_removable_rows']}")

cardiac - ultra-rare: High - 9, Low - 2; removed_rows_from_random: 11
cardiac - singleton: High - 9, Low - 2; removed_rows_from_random: 11
cava - ultra-rare: High - 38, Low - 8; removed_rows_from_random: 46
cava - singleton: High - 38, Low - 8; removed_rows_from_random: 46
neuro - ultra-rare: High - 4, Low - 1; removed_rows_from_random: 5
neuro - singleton: High - 4, Low - 1; removed_rows_from_random: 5
random - ultra-rare: High - 28, Low - 6; removed_rows_from_random: 34
random - singleton: High - 28, Low - 6; removed_rows_from_random: 34


In [13]:
resampling_results

{'cardiac': {'ultra-rare': {'resample_number_high': 9,
   'resample_number_low': 2,
   'random_removable_rows': 11},
  'singleton': {'resample_number_high': 9,
   'resample_number_low': 2,
   'random_removable_rows': 11}},
 'cava': {'ultra-rare': {'resample_number_high': 38,
   'resample_number_low': 8,
   'random_removable_rows': 46},
  'singleton': {'resample_number_high': 38,
   'resample_number_low': 8,
   'random_removable_rows': 46}},
 'neuro': {'ultra-rare': {'resample_number_high': 4,
   'resample_number_low': 1,
   'random_removable_rows': 5},
  'singleton': {'resample_number_high': 4,
   'resample_number_low': 1,
   'random_removable_rows': 5}},
 'random': {'ultra-rare': {'resample_number_high': 28,
   'resample_number_low': 6,
   'random_removable_rows': 34},
  'singleton': {'resample_number_high': 28,
   'resample_number_low': 6,
   'random_removable_rows': 34}}}

#### Look on the enformer_class numbers and compare them to the numbers after the subsampling

#### Subsample

In [14]:
import pandas as pd

def update_enformer_classes(df, resampling_info):
    new_rows = []
    for gene_set, variant_info in resampling_info.items():
        for variant_type, sampling_info in variant_info.items():
            print(f"Processing gene_set: {gene_set}, variant_type: {variant_type}")
            
            # 1. Get a DataFrame of the gene_set and variant_type
            subset_df = df[(df['gene_set'] == gene_set) & (df['variant_type'] == variant_type)]
            print(f"Initial subset size: {subset_df.shape[0]}")
            
            # 2. For the enformer_random set: set the enformer_class to 'old_enformer_random' for random_removable_rows rows
            random_removable_rows = sampling_info['random_removable_rows']
            if random_removable_rows > 0:
                random_indices = subset_df[subset_df['enformer_class'] == 'enformer_random'].sample(n=random_removable_rows, replace=False, random_state=1313).index
                df.loc[random_indices, 'enformer_class'] = 'old_enformer_random'
                print(f"Set {random_removable_rows} enformer_random rows to old_enformer_random")

            # 3. Sample from enformer_high and enformer_low
            resample_number_high = sampling_info['resample_number_high']
            resample_number_low = sampling_info['resample_number_low']
            
            if resample_number_high > 0:
                high_sample = subset_df[subset_df['enformer_class'] == 'enformer_high'].sample(n=resample_number_high, replace=False, random_state=1313)
                high_sample['enformer_class'] = 'enformer_random'
                new_rows.append(high_sample)
                print(f"Sampled {resample_number_high} from enformer_high to be added as enformer_random")
                
            if resample_number_low > 0:
                low_sample = subset_df[subset_df['enformer_class'] == 'enformer_low'].sample(n=resample_number_low, replace=False, random_state=1313)
                low_sample['enformer_class'] = 'enformer_random'
                new_rows.append(low_sample)
                print(f"Sampled {resample_number_low} from enformer_low to be added as enformer_random")
    
    # Concatenate the new rows to the original DataFrame
    if new_rows:
        new_df = pd.concat(new_rows)
        df = pd.concat([df, new_df], ignore_index=True)

    return df


print("Before update:")
before_counts = df_exploded.groupby(['gene_set', 'variant_type', 'enformer_class']).size().reset_index(name='counts')
print(before_counts)

updated_df = update_enformer_classes(df_exploded, resampling_results)
# print(updated_df)
print("\nAfter update:")
after_counts = updated_df.groupby(['gene_set', 'variant_type', 'enformer_class']).size().reset_index(name='counts')
print(after_counts)

Before update:
   gene_set variant_type   enformer_class  counts
0   cardiac    singleton    enformer_high    3500
1   cardiac    singleton     enformer_low     750
2   cardiac    singleton  enformer_random     750
3   cardiac   ultra-rare    enformer_high    3500
4   cardiac   ultra-rare     enformer_low     750
5   cardiac   ultra-rare  enformer_random     750
6      cava    singleton    enformer_high    3500
7      cava    singleton     enformer_low     750
8      cava    singleton  enformer_random     750
9      cava   ultra-rare    enformer_high    3500
10     cava   ultra-rare     enformer_low     750
11     cava   ultra-rare  enformer_random     750
12    neuro    singleton    enformer_high    3500
13    neuro    singleton     enformer_low     750
14    neuro    singleton  enformer_random     750
15    neuro   ultra-rare    enformer_high    3500
16    neuro   ultra-rare     enformer_low     750
17    neuro   ultra-rare  enformer_random     750
18   random    singleton    enforme

###### Old code not following the correct way to do the subsampling

In [15]:
# # Set random seed
# np.random.seed(1313)

# # Function to update the enformer_class based on sampling
# def update_enformer_class(df, gene_set, variant_type, resample_number_high, resample_number_low):
#     # Filter the DataFrame for the specific subset
#     subset_df = df[(df['gene_set'] == gene_set) & (df['variant_type'] == variant_type)]

#     # Calculate the number of rows to sample for enformer_high and enformer_low
#     num_high = int(resample_number_high)
#     num_low = int(resample_number_low)
    
#     # Sample rows
#     high_indices = subset_df[subset_df['enformer_class'] == 'enformer_high'].sample(n=num_high, replace=False, random_state=1313).index
#     low_indices = subset_df[subset_df['enformer_class'] == 'enformer_low'].sample(n=num_low, replace=False, random_state=1313).index
    
#     # Update the enformer_class
#     df.loc[high_indices, 'enformer_class'] = 'enformer_random'
#     df.loc[low_indices, 'enformer_class'] = 'enformer_random'
    
# # Count the enformer_class occurrences before the update
# # print("Before update:")
# before_counts = df_exploded.groupby(['gene_set', 'variant_type', 'enformer_class']).size().reset_index(name='counts')
# # print(before_counts)

# # Apply the function to all gene_set and variant_type combinations
# for gene_set in results.keys():
#     for variant_type in results[gene_set].keys():
#         resample_number_high = results[gene_set][variant_type]['resample_number_high']
#         resample_number_low = results[gene_set][variant_type]['resample_number_low']
#         update_enformer_class(df_exploded, gene_set, variant_type, resample_number_high, resample_number_low)

# # Count the enformer_class occurrences after the update
# # print("\nAfter update:")
# after_counts = df_exploded.groupby(['gene_set', 'variant_type', 'enformer_class']).size().reset_index(name='counts')
# # print(after_counts)


# # Display the updated DataFrame
# # print(df_exploded)

# # Expected after update (if run multiple times the process is done multiple times)
# # After update:
# #    gene_set variant_type   enformer_class  counts
# # 0   cardiac    singleton    enformer_high    2712
# # 1   cardiac    singleton     enformer_low     575
# # 2   cardiac    singleton  enformer_random    1713
# # 3   cardiac   ultra-rare    enformer_high    2712
# # 4   cardiac   ultra-rare     enformer_low     576
# # 5   cardiac   ultra-rare  enformer_random    1712
# # 6      cava    singleton    enformer_high    2725
# # 7      cava    singleton     enformer_low     582
# # 8      cava    singleton  enformer_random    1693
# # 9      cava   ultra-rare    enformer_high    2732
# # 10     cava   ultra-rare     enformer_low     578
# # 11     cava   ultra-rare  enformer_random    1690
# # 12    neuro    singleton    enformer_high    2694
# # 13    neuro    singleton     enformer_low     587
# # 14    neuro    singleton  enformer_random    1719
# # 15    neuro   ultra-rare    enformer_high    2701
# # 16    neuro   ultra-rare     enformer_low     579
# # 17    neuro   ultra-rare  enformer_random    1720
# # 18   random    singleton    enformer_high    1376
# # 19   random    singleton     enformer_low     295
# # 20   random    singleton  enformer_random     829
# # 21   random   ultra-rare    enformer_high    1376
# # 22   random   ultra-rare     enformer_low     295
# # 23   random   ultra-rare  enformer_random     829
 

In [16]:
after_counts

,gene_set,variant_type,enformer_class,counts
0,cardiac,singleton,enformer_high,3500
1,cardiac,singleton,enformer_low,750
2,cardiac,singleton,enformer_random,750
3,cardiac,singleton,old_enformer_random,11
4,cardiac,ultra-rare,enformer_high,3500
5,cardiac,ultra-rare,enformer_low,750
6,cardiac,ultra-rare,enformer_random,750
7,cardiac,ultra-rare,old_enformer_random,11
8,cava,singleton,enformer_high,3500
9,cava,singleton,enformer_low,750


In [17]:
# investigate resampling results
resampling_results 
# => expected enformer high and enformer low: 
2*9 + 2* 38 + 2* 4 + 2*28 # 158
2*2 + 2* 8 + 2*1 + 2*6# 34


34

In [18]:
updated_df.head()

,CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,chr_pos_ref_alt,DNase_max,max_col,enformer_variant_info,gene_set,variant_type,enformer_class
0,chr6,43797164,VEGFA|ENSG00000112715.26|EH38E3709352|6-437971...,G,A,1,PASS,AF=6.57168e-06;AC=1,6-43797164-G-A,4254.3410,392_DNASE:CD14-positive,"(cardiac, singleton, enformer_high)",cardiac,singleton,enformer_high
1,chr18,3059465,MYOM1|ENSG00000101605.14|EH38E3253954|18-30594...,T,C,1,PASS,AF=6.56866e-06;AC=1,18-3059465-T-C,4022.5996,"177_DNASE:CD8-positive,","(cardiac, singleton, enformer_high)",cardiac,singleton,enformer_high
2,chr6,43797016,VEGFA|ENSG00000112715.26|EH38E3709352|6-437970...,G,A,1,PASS,AF=6.57618e-06;AC=1,6-43797016-G-A,3591.8310,392_DNASE:CD14-positive,"(cardiac, singleton, enformer_high)",cardiac,singleton,enformer_high
3,chr12,2616967,CACNA1C|ENSG00000151067.23|EH38E2998634|12-261...,C,A,1,PASS,AF=6.56763e-06;AC=1,12-2616967-C-A,2754.7705,401_DNASE:common myeloid,"(cardiac, singleton, enformer_high)",cardiac,singleton,enformer_high
4,chr12,2616967,CACNA1C|ENSG00000151067.23|EH38E2998634|12-261...,C,A,1,PASS,AF=6.56763e-06;AC=1,12-2616967-C-A,2754.7705,401_DNASE:common myeloid,"(neuro, singleton, enformer_high)",neuro,singleton,enformer_high


In [19]:
print('All: ', updated_df.shape[0])
updated_df['chr_pos_ref_alt'].nunique() # not unique
# number of unique variant positions in informer class: if we match them they get counted multiple times
updated_df.loc[updated_df['enformer_class'] == 'enformer_random']['chr_pos_ref_alt'].nunique() # 5250 => all unqiue

All:  35192


5250

In [20]:
updated_df.columns
updated_df.enformer_class.value_counts()

enformer_class
enformer_high          24500
enformer_random         5250
enformer_low            5250
old_enformer_random      192
Name: count, dtype: int64

#### For the downstram analysis I need to identify at which DNase_max value enformer decided to call it high or low
- the minimal values of DNase_max for enformer high range from 96 (random, ultra-rare) to 409 (neuro, singleton). One can observe that the miniaml prediction is higher for the singleton than for the ultra-rare in each gene set
- the maximal values of DNase_max for enformer low range from 0.8 (neuro, singleton) to 1.98 (random, ultra-rare) while all ultra-rare groups have a higher maximal predicted DNase_max value
- compute these values from the initial data frame and not from the changed one (changing values and then computing thresholds makes no sense)

In [2]:
common_enformer_varaint_df = pd.read_csv(config['files']['creating']['common_and_enformer_variants'], sep="\t")
common_enformer_varaint_df['enformer_variant_info'] = common_enformer_varaint_df['enformer_variant_info'].apply(literal_eval)
# common_enformer_varaint_df['enformer_class_list'] = common_enformer_varaint_df['enformer_variant_info'].apply(hf.get_enformer_class_list)
# common_enformer_varaint_df['gene_set_list'] = common_enformer_varaint_df['enformer_variant_info'].apply(hf.get_gene_set_list)
# common_enformer_varaint_df['variant_type_list'] = common_enformer_varaint_df['enformer_variant_info'].apply(hf.get_variant_type_list)

enformer_class_df = common_enformer_varaint_df.loc[~common_enformer_varaint_df['DNase_max'].isna()]

# Explode the list of tuples into separate rows
df_initial_exploded = enformer_class_df.explode('enformer_variant_info')

# Split the tuples into separate columns
df_initial_exploded[['gene_set', 'variant_type', 'enformer_class']] = pd.DataFrame(df_initial_exploded['enformer_variant_info'].tolist(), index=df_initial_exploded.index)

NameError: name 'pd' is not defined

In [22]:
# Filter for enformer_class='enformer_high'
df_high = df_initial_exploded[df_initial_exploded['enformer_class'] == 'enformer_high']

# Group by gene_set and variant_type, and find the minimum DNase_max
min_dnase = df_high.groupby(['gene_set', 'variant_type'])['DNase_max'].min().reset_index()
min_dnase = min_dnase.rename(columns={'DNase_max': 'high_min_DNase_max'})

df_enformer_low = df_initial_exploded[df_initial_exploded['enformer_class'] == 'enformer_low']
max_dnase = df_enformer_low.groupby(['gene_set', 'variant_type'])['DNase_max'].max().reset_index()
max_dnase = max_dnase.rename(columns={'DNase_max': 'low_max_DNase_max'})


In [23]:
min_dnase

,gene_set,variant_type,high_min_DNase_max
0,cardiac,singleton,304.926800
1,cardiac,ultra-rare,274.176540
2,cava,singleton,123.103165
3,cava,ultra-rare,108.784100
4,neuro,singleton,409.231380
5,neuro,ultra-rare,381.130000
6,random,singleton,105.228670
7,random,ultra-rare,96.032210


In [24]:
max_dnase

,gene_set,variant_type,low_max_DNase_max
0,cardiac,singleton,1.186899
1,cardiac,ultra-rare,1.227749
2,cava,singleton,1.585564
3,cava,ultra-rare,1.634381
4,neuro,singleton,0.803452
5,neuro,ultra-rare,0.824037
6,random,singleton,1.898443
7,random,ultra-rare,1.991982


#### Compute the number of enformer_random above the threshold
- I can merge the enformer random df with the thresholds and look into the number of rows which are higher the DNase_max

In [25]:
5000 * 0.15 

750.0

In [26]:
# example for random set:
print(1750 - 375) # enformer_high: 
print(80 + 375 + 375) # enformer_random
print(375 - 80) # enformer_low
# is as expected see below

1375
830
295


In [27]:
updated_df.groupby(by=['gene_set', 'variant_type'])['enformer_class'].value_counts()

gene_set  variant_type  enformer_class     
cardiac   singleton     enformer_high          3500
                        enformer_low            750
                        enformer_random         750
                        old_enformer_random      11
          ultra-rare    enformer_high          3500
                        enformer_low            750
                        enformer_random         750
                        old_enformer_random      11
cava      singleton     enformer_high          3500
                        enformer_low            750
                        enformer_random         750
                        old_enformer_random      46
          ultra-rare    enformer_high          3500
                        enformer_low            750
                        enformer_random         750
                        old_enformer_random      46
neuro     singleton     enformer_high          3500
                        enformer_low            750
                    

In [28]:
updated_df

,CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,chr_pos_ref_alt,DNase_max,max_col,enformer_variant_info,gene_set,variant_type,enformer_class
0,chr6,43797164,VEGFA|ENSG00000112715.26|EH38E3709352|6-437971...,G,A,1,PASS,AF=6.57168e-06;AC=1,6-43797164-G-A,4254.341000,392_DNASE:CD14-positive,"(cardiac, singleton, enformer_high)",cardiac,singleton,enformer_high
1,chr18,3059465,MYOM1|ENSG00000101605.14|EH38E3253954|18-30594...,T,C,1,PASS,AF=6.56866e-06;AC=1,18-3059465-T-C,4022.599600,"177_DNASE:CD8-positive,","(cardiac, singleton, enformer_high)",cardiac,singleton,enformer_high
2,chr6,43797016,VEGFA|ENSG00000112715.26|EH38E3709352|6-437970...,G,A,1,PASS,AF=6.57618e-06;AC=1,6-43797016-G-A,3591.831000,392_DNASE:CD14-positive,"(cardiac, singleton, enformer_high)",cardiac,singleton,enformer_high
3,chr12,2616967,CACNA1C|ENSG00000151067.23|EH38E2998634|12-261...,C,A,1,PASS,AF=6.56763e-06;AC=1,12-2616967-C-A,2754.770500,401_DNASE:common myeloid,"(cardiac, singleton, enformer_high)",cardiac,singleton,enformer_high
4,chr12,2616967,CACNA1C|ENSG00000151067.23|EH38E2998634|12-261...,C,A,1,PASS,AF=6.56763e-06;AC=1,12-2616967-C-A,2754.770500,401_DNASE:common myeloid,"(neuro, singleton, enformer_high)",neuro,singleton,enformer_high
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35187,chr2,74459935,MOGS|ENSG00000115275.15|EH38E3353079|2-7445993...,C,T,1,PASS,AF=6.57022e-06;AC=1,2-74459935-C-T,1.396676,5_DNASE:GM03348 geneti,"(random, singleton, enformer_low)",random,singleton,enformer_random
35188,chr10,79586224,SFTPA1|ENSG00000122852.15|EH38E2912112|10-7958...,G,T,1,PASS,AF=6.56953e-06;AC=1,10-79586224-G-T,1.358134,126_DNASE:LNCaP clone FG,"(random, singleton, enformer_low)",random,singleton,enformer_random
35189,chr14,24475326,CMA1|ENSG00000092009.10|EH38E3087811|14-244753...,C,G,1,PASS,AF=6.57272e-06;AC=1,14-24475326-C-G,1.607328,20_DNASE:H7-hESC,"(random, singleton, enformer_low)",random,singleton,enformer_random
35190,chr12,48745122,TEX49|ENSG00000257987.6|EH38E3014983|12-487451...,T,A,1,PASS,AF=6.61664e-06;AC=1,12-48745122-T-A,1.056721,392_DNASE:CD14-positive,"(random, singleton, enformer_low)",random,singleton,enformer_random


In [29]:
random_df = updated_df[updated_df['enformer_class'] == 'enformer_random']
print('Number of random variants after resampling: ', random_df.shape[0])
# Merge with min_dnase to apply the threshold for enformer_high
random_df = pd.merge(random_df, min_dnase, on=['gene_set', 'variant_type'], how='left')
random_df = pd.merge(random_df, max_dnase, on=['gene_set', 'variant_type'], how='left')


# Count the number of enformer_random rows with DNase_max above the min_dnase thresholds
count_enformer_high = random_df[random_df['DNase_max'] >= random_df['high_min_DNase_max']].shape[0]

# Count the number of enformer_random rows with DNase_max below the max_dnase thresholds
count_enformer_low = random_df[random_df['DNase_max'] <= random_df['low_max_DNase_max']].shape[0]

print(f"Number of enformer_random rows with DNase_max above the min_dnase thresholds: {count_enformer_high}")
print(f"Number of enformer_random rows with DNase_max below the max_dnase thresholds: {count_enformer_low}")

Number of random variants after resampling:  5250
Number of enformer_random rows with DNase_max above the min_dnase thresholds: 158
Number of enformer_random rows with DNase_max below the max_dnase thresholds: 34


### Just get for each group of gene_set and variant_type the number of randoms above the group specific enformer threshold of being high or low
1. Split the dataframe of interest into only considering enformer predicted variants
2. Compute the group specific thresholds
3. For each group get the new random set
4. Compute the number of variants above or below the enformer thresholds (left join the thresholds to the random df and count)

In [4]:
common_enformer_varaint_df = pd.read_csv(config['files']['creating']['common_and_enformer_variants'], sep="\t")
common_enformer_varaint_df['enformer_variant_info'] = common_enformer_varaint_df['enformer_variant_info'].apply(literal_eval)
# common_enformer_varaint_df['enformer_class_list'] = common_enformer_varaint_df['enformer_variant_info'].apply(hf.get_enformer_class_list)
# common_enformer_varaint_df['gene_set_list'] = common_enformer_varaint_df['enformer_variant_info'].apply(hf.get_gene_set_list)
# common_enformer_varaint_df['variant_type_list'] = common_enformer_varaint_df['enformer_variant_info'].apply(hf.get_variant_type_list)

enformer_class_df = common_enformer_varaint_df.loc[~common_enformer_varaint_df['DNase_max'].isna()]

In [5]:
# Explode the list of tuples into separate rows
df_initial_exploded = enformer_class_df.explode('enformer_variant_info')

# Split the tuples into separate columns
df_initial_exploded[['gene_set', 'variant_type', 'enformer_class']] = pd.DataFrame(df_initial_exploded['enformer_variant_info'].tolist(), index=df_initial_exploded.index)

# Filter for enformer_class='enformer_high'
df_high = df_initial_exploded[df_initial_exploded['enformer_class'] == 'enformer_high']

# Group by gene_set and variant_type, and find the minimum DNase_max of enformer_high and the max DNase_max of enformer_low 
min_dnase = df_high.groupby(['gene_set', 'variant_type'])['DNase_max'].min().reset_index()
min_dnase = min_dnase.rename(columns={'DNase_max': 'high_min_DNase_max'})

df_enformer_low = df_initial_exploded[df_initial_exploded['enformer_class'] == 'enformer_low']
max_dnase = df_enformer_low.groupby(['gene_set', 'variant_type'])['DNase_max'].max().reset_index()
max_dnase = max_dnase.rename(columns={'DNase_max': 'low_max_DNase_max'})

In [32]:
df_initial_exploded.loc[df_initial_exploded.duplicated(['CHROM', 'POS','REF','ALT'], keep=False)].sort_values(by='POS') # 900 with keep=False: 1759

,CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,chr_pos_ref_alt,DNase_max,max_col,enformer_variant_info,gene_set,variant_type,enformer_class
35988,chr11,492959,HRAS|ENSG00000174775.18|EH38E2937561|11-492959...,C,T,1,PASS,AF=6.57834e-06;AC=1,11-492959-C-T,411.52228,408_DNASE:OCI-LY7,"(cardiac, singleton, enformer_high)",cardiac,singleton,enformer_high
35988,chr11,492959,HRAS|ENSG00000174775.18|EH38E2937561|11-492959...,C,T,1,PASS,AF=6.57834e-06;AC=1,11-492959-C-T,411.52228,408_DNASE:OCI-LY7,"(neuro, singleton, enformer_high)",neuro,singleton,enformer_high
34414,chr11,501161,HRAS|ENSG00000174775.18|EH38E2937583|11-501161...,G,T,1,PASS,AF=6.57212e-06;AC=1,11-501161-G-T,866.44324,238_DNASE:heart female e,"(cardiac, singleton, enformer_high)",cardiac,singleton,enformer_high
34414,chr11,501161,HRAS|ENSG00000174775.18|EH38E2937583|11-501161...,G,T,1,PASS,AF=6.57212e-06;AC=1,11-501161-G-T,866.44324,238_DNASE:heart female e,"(neuro, singleton, enformer_high)",neuro,singleton,enformer_high
58377,chr11,502604,HRAS|ENSG00000174775.18|EH38E2937588|11-502604...,A,C,1,PASS,AF=7.89567e-05;AC=12,11-502604-A-C,0.76589,13_DNASE:GM12891,"(cardiac, ultra-rare, enformer_low)",cardiac,ultra-rare,enformer_low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
35039,chr1,156165278,LMNA|ENSG00000160789.24|EH38E2840582|1-1561652...,C,A,1,PASS,AF=6.57212e-06;AC=1,1-156165278-C-A,557.32697,392_DNASE:CD14-positive,"(cava, singleton, enformer_high)",cava,singleton,enformer_high
35056,chr1,156165340,LMNA|ENSG00000160789.24|EH38E2840582|1-1561653...,C,T,1,PASS,AF=6.57895e-06;AC=1,1-156165340-C-T,552.71960,392_DNASE:CD14-positive,"(cardiac, singleton, enformer_high)",cardiac,singleton,enformer_high
35056,chr1,156165340,LMNA|ENSG00000160789.24|EH38E2840582|1-1561653...,C,T,1,PASS,AF=6.57895e-06;AC=1,1-156165340-C-T,552.71960,392_DNASE:CD14-positive,"(cava, singleton, enformer_high)",cava,singleton,enformer_high
35579,chr1,156178444,LMNA|ENSG00000160789.24|EH38E1387748|1-1561784...,A,C,1,PASS,AF=6.57618e-06;AC=1,1-156178444-A-C,461.86017,"177_DNASE:CD8-positive,","(cardiac, singleton, enformer_high)",cardiac,singleton,enformer_high


Split the df group specific and compute the new random set for each group

In [33]:
total_number_dict = {'cardiac': [('ultra-rare',289724), ('singleton', 296173)],
                     'cava': [('ultra-rare',69363), ('singleton', 69286)],
                     'neuro': [('ultra-rare',598537), ('singleton', 607227)],
                     'random': [('ultra-rare',23216), ('singleton', 24397)],
                    } # from Y1_MPRA_design.pptx

prioritized_variant_number = {'cardiac': [('ultra-rare',5000), ('singleton', 5000)],
                     'cava': [('ultra-rare',5000), ('singleton', 5000)],
                     'neuro': [('ultra-rare',5000), ('singleton', 5000)],
                     'random': [('ultra-rare',2500), ('singleton', 2500)],
                    }

def get_new_random_group(df, gene_set, variant_type, high_ratio=0.7, random_ratio=0.15, low_ratio=0.15, random_seed=1313): 
    """Split the dataframe and resample the enformer_random from enformer_high and enformer_low return only the new enformer_random df"""
    # Split group
    subset_df = df[(df['gene_set'] == gene_set) & (df['variant_type'] == variant_type)]
    # print(subset_df[['gene_set', 'variant_type', 'enformer_class']].groupby(by=['gene_set', 'variant_type'])['enformer_class'].value_counts())
    
    chosen_variants = prioritized_variant_number[gene_set][0][1]
    total_number = total_number_dict[gene_set][0][1]
    
    # compute sampling ration (#RANDOM / (#ALL VARIANTS - #HIGH - #LOW))
    not_random_num = chosen_variants * (high_ratio + low_ratio)
    sampling_ration_random = (chosen_variants * random_ratio) / (total_number - not_random_num)
    random_sampling_num = round(sampling_ration_random * not_random_num)
    high_sampling_num = round(sampling_ration_random * (chosen_variants * high_ratio))
    low_sampling_num = random_sampling_num - high_sampling_num
    # print(high_sampling_num, random_sampling_num, low_sampling_num)
    
    # sample from high and low based on index and change enformer class     
    low_sample_index = subset_df[subset_df['enformer_class'] == 'enformer_low'].sample(n=low_sampling_num, replace=False, random_state=random_seed).index
    high_sample_index = subset_df[subset_df['enformer_class'] == 'enformer_high'].sample(n=high_sampling_num, replace=False, random_state=random_seed).index
    
    # Update enformer_class
    subset_df.loc[high_sample_index, 'enformer_class'] = 'enformer_random'
    subset_df.loc[low_sample_index, 'enformer_class'] = 'enformer_random'
    # Check counts of enformer class again
    # print(subset_df[['gene_set', 'variant_type', 'enformer_class']].groupby(by=['gene_set', 'variant_type'])['enformer_class'].value_counts())

    # only interested in enformer_random
    random_df = subset_df.loc[subset_df['enformer_class'] == 'enformer_random']
    print(f'Number of random set for {gene_set} and {variant_type} {random_df.shape[0]}')
    return random_df

def investigate_random_df(random_df, high_enformer_threshold_df, low_enformer_threshold_df):
    # Merge with min_dnase to apply the threshold for enformer_high
    current_random_df = pd.merge(random_df, high_enformer_threshold_df, on=['gene_set', 'variant_type'], how='left')
    current_random_df = pd.merge(current_random_df, low_enformer_threshold_df, on=['gene_set', 'variant_type'], how='left')

    # Count the number of enformer_random rows with DNase_max above the min_dnase thresholds
    count_enformer_high = current_random_df[current_random_df['DNase_max'] >= current_random_df['high_min_DNase_max']].shape[0]

    # Count the number of enformer_random rows with DNase_max below the max_dnase thresholds
    count_enformer_low = current_random_df[current_random_df['DNase_max'] <= current_random_df['low_max_DNase_max']].shape[0]
    print(f"Number of enformer_random rows with DNase_max above the min_dnase thresholds: {count_enformer_high}")
    print(f"Number of enformer_random rows with DNase_max below the max_dnase thresholds: {count_enformer_low}")
    return count_enformer_high, count_enformer_low
    

In [64]:
df_initial_exploded

,CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,chr_pos_ref_alt,DNase_max,max_col,enformer_variant_info,gene_set,variant_type,enformer_class
34122,chr6,43797164,VEGFA|ENSG00000112715.26|EH38E3709352|6-437971...,G,A,1,PASS,AF=6.57168e-06;AC=1,6-43797164-G-A,4254.341000,392_DNASE:CD14-positive,"(cardiac, singleton, enformer_high)",cardiac,singleton,enformer_high
34123,chr18,3059465,MYOM1|ENSG00000101605.14|EH38E3253954|18-30594...,T,C,1,PASS,AF=6.56866e-06;AC=1,18-3059465-T-C,4022.599600,"177_DNASE:CD8-positive,","(cardiac, singleton, enformer_high)",cardiac,singleton,enformer_high
34124,chr6,43797016,VEGFA|ENSG00000112715.26|EH38E3709352|6-437970...,G,A,1,PASS,AF=6.57618e-06;AC=1,6-43797016-G-A,3591.831000,392_DNASE:CD14-positive,"(cardiac, singleton, enformer_high)",cardiac,singleton,enformer_high
34125,chr12,2616967,CACNA1C|ENSG00000151067.23|EH38E2998634|12-261...,C,A,1,PASS,AF=6.56763e-06;AC=1,12-2616967-C-A,2754.770500,401_DNASE:common myeloid,"(cardiac, singleton, enformer_high)",cardiac,singleton,enformer_high
34125,chr12,2616967,CACNA1C|ENSG00000151067.23|EH38E2998634|12-261...,C,A,1,PASS,AF=6.56763e-06;AC=1,12-2616967-C-A,2754.770500,401_DNASE:common myeloid,"(neuro, singleton, enformer_high)",neuro,singleton,enformer_high
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
68217,chr2,219219689,ANKZF1|ENSG00000163516.14|EH38E3401197|2-21921...,T,G,1,PASS,AF=1.97075e-05;AC=3,2-219219689-T-G,-0.586787,31_DNASE:iPS-NIHi11 mal,"(random, ultra-rare, enformer_low)",random,ultra-rare,enformer_low
68218,chr19,48747089,IZUMO1|ENSG00000182264.9|EH38E1960256|19-48747...,C,T,1,PASS,AF=1.31413e-05;AC=2,19-48747089-C-T,-0.961862,90_DNASE:HeLa-S3 G1b ph,"(random, ultra-rare, enformer_low)",random,ultra-rare,enformer_low
68219,chr2,210475933,CPS1|ENSG00000021826.18|EH38E3397490|2-2104759...,C,A,1,PASS,AF=2.62871e-05;AC=4,2-210475933-C-A,-0.964058,665_DNASE:testis male ad,"(random, ultra-rare, enformer_low)",random,ultra-rare,enformer_low
68220,chr2,210477091,CPS1|ENSG00000021826.18|EH38E3397493|2-2104770...,C,T,1,PASS,AF=7.60025e-05;AC=11,2-210477091-C-T,-1.122092,170_DNASE:iPS DF 4.7 mal,"(random, ultra-rare, enformer_low)",random,ultra-rare,enformer_low


In [34]:
all_random_dfs = []
for gene_set in total_number_dict.keys():
    for variant_type, _ in total_number_dict[gene_set]:
        new_random_df = get_new_random_group(df_initial_exploded, gene_set=gene_set, variant_type=variant_type, high_ratio=0.7, random_ratio=0.15, low_ratio=0.15)
        all_random_dfs.append(new_random_df)

new_random_classes = pd.concat([df for df in all_random_dfs])
print('Shape: ', new_random_classes.shape[0])
print('Unique: ', new_random_classes.chr_pos_ref_alt.nunique())

Number of random set for cardiac and ultra-rare 761
Number of random set for cardiac and singleton 761
Number of random set for cava and ultra-rare 799
Number of random set for cava and singleton 799
Number of random set for neuro and ultra-rare 755
Number of random set for neuro and singleton 755
Number of random set for random and ultra-rare 413
Number of random set for random and singleton 413
Shape:  5456
Unique:  5456


In [35]:
# write only random class df into table
new_random_outpath = '/home/kisa/coding/80K_MPRA/enformer_data/only_random_enformer/random_variants.tsv'
# # # new_random_outpath = know what you are doing!! '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/projects/enformer_predictions/random_variants.tsv'
# new_random_outpath = '/data/gpfs-1/users/kisa11_c/work/coding/MPRA/IGVF_Y1_design/projects/enformer_predictions/random_variants_new_version.tsv'
# new_random_classes.to_csv(new_random_outpath, sep="\t", index=False)

#### add the neuro specific information sort the dataframe and only hold the 101 first (how many are significant from these?) 
- do this in `80K-Analysis/06_variant_analysis/notebooks/thoughtful_way_annotating_MPRAlm.ipynb` (because has more information and code for this problem)

In [36]:
def is_significant(p_value):
    return 'yes' if p_value < 0.05 else 'no'

In [37]:
# read the variant table 
bcMPRAlm_df = pd.read_csv(config['files']['creating']['toptable_bcMPRAlm_resequencing'], sep="\t")
bcMPRAlm_df
# only tested: 
tested_bcMPRAlm_df = bcMPRAlm_df.loc[bcMPRAlm_df['variant_id'].str.startswith('cardiac_neuro_cava_random')]
tested_bcMPRAlm_df
# add interesting columns
tested_bcMPRAlm_df['chr_pos_ref_alt'] = tested_bcMPRAlm_df['variant_id'].apply(hf.get_chrom_pos_ref_alt_pattern) # result is unique
tested_bcMPRAlm_df['is_significant'] = tested_bcMPRAlm_df['adj.P.Val'].apply(is_significant)
tested_bcMPRAlm_df.head() # logFC	AveExpr	t	P.Value	adj.P.Val	B	variant_id	chr_pos_ref_alt


/tmp/ipykernel_55402/3565532276.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tested_bcMPRAlm_df['chr_pos_ref_alt'] = tested_bcMPRAlm_df['variant_id'].apply(hf.get_chrom_pos_ref_alt_pattern) # result is unique
/tmp/ipykernel_55402/3565532276.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tested_bcMPRAlm_df['is_significant'] = tested_bcMPRAlm_df['adj.P.Val'].apply(is_significant)


,logFC,AveExpr,t,P.Value,adj.P.Val,B,variant_id,chr_pos_ref_alt,is_significant
0,1.482729,1.893987,19.158362,2.947206e-66,1.081448e-61,138.643345,cardiac_neuro_cava_random:TRIO|ENSG00000038382...,5-14408059-A-G,yes
1,1.521560,0.992996,13.599182,1.398939e-35,2.566633e-31,68.719277,cardiac_neuro_cava_random:TRIO|ENSG00000038382...,5-14259916-C-T,yes
2,1.489524,0.944281,13.279882,6.660143e-34,8.146243e-30,64.966845,cardiac_neuro_cava_random:ANKZF1|ENSG000001635...,2-219267001-T-C,yes
3,0.870725,0.957196,12.290067,3.299979e-32,3.027235e-28,62.207371,cardiac_neuro_cava_random:ATR|ENSG00000175054....,3-142578095-C-G,yes
4,1.027577,1.135585,12.214664,6.337828e-31,4.651206e-27,59.078895,cardiac_neuro_cava_random:CDH1|ENSG00000039068...,16-68784390-A-G,yes


In [38]:
# remove columns
tested_bcMPRAlm_df = tested_bcMPRAlm_df.drop(columns=['AveExpr', 't', 'B'])
tested_bcMPRAlm_df.columns

Index(['logFC', 'P.Value', 'adj.P.Val', 'variant_id', 'chr_pos_ref_alt',
       'is_significant'],
      dtype='object')

In [39]:
new_random_classes.columns

Index(['CHROM', 'POS', 'ID', 'REF', 'ALT', 'QUAL', 'FILTER', 'INFO',
       'chr_pos_ref_alt', 'DNase_max', 'max_col', 'enformer_variant_info',
       'gene_set', 'variant_type', 'enformer_class'],
      dtype='object')

In [40]:
new_random_classes

,CHROM,POS,ID,REF,ALT,QUAL,FILTER,INFO,chr_pos_ref_alt,DNase_max,max_col,enformer_variant_info,gene_set,variant_type,enformer_class
53911,chr18,31490614,DSG2|ENSG00000046604.14|EH38E1907516|18-314906...,C,T,1,PASS,AF=0.00816117;AC=1242,18-31490614-C-T,873.882600,284_DNASE:ELR,"(cardiac, ultra-rare, enformer_high)",cardiac,ultra-rare,enformer_random
55139,chr1,164794455,PBX1|ENSG00000185630.20|EH38E1393315|1-1647944...,G,A,1,PASS,AF=1.31413e-05;AC=2,1-164794455-G-A,410.446000,98_DNASE:fibroblast of,"(cardiac, ultra-rare, enformer_high)",cardiac,ultra-rare,enformer_random
55180,chr17,17840972,SREBF1|ENSG00000072310.18|EH38E3212322|17-1784...,C,T,1,PASS,AF=1.31368e-05;AC=2,17-17840972-C-T,405.337860,392_DNASE:CD14-positive,"(cardiac, ultra-rare, enformer_high)",cardiac,ultra-rare,enformer_random
55480,chr19,4052024,MAP2K2|ENSG00000126934.15|EH38E3284367|19-4052...,G,C,1,PASS,AF=1.97267e-05;AC=3,19-4052024-G-C,373.998380,392_DNASE:CD14-positive,"(cardiac, ultra-rare, enformer_high)",cardiac,ultra-rare,enformer_random
55857,chr17,16297226,PIGL|ENSG00000108474.17|EH38E1849289|17-162972...,C,G,1,PASS,AF=3.28528e-05;AC=5,17-16297226-C-G,342.911300,234_DNASE:HepG2,"(cardiac, ultra-rare, enformer_high)",cardiac,ultra-rare,enformer_random
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
65482,chr22,32100959,SLC5A1|ENSG00000100170.10|EH38E2159120|22-3210...,C,G,1,PASS,AF=6.578e-06;AC=1,22-32100959-C-G,1.544069,"514_DNASE:CD4-positive,","(random, singleton, enformer_low)",random,singleton,enformer_random
65525,chr2,74459935,MOGS|ENSG00000115275.15|EH38E3353079|2-7445993...,C,T,1,PASS,AF=6.57022e-06;AC=1,2-74459935-C-T,1.396676,5_DNASE:GM03348 geneti,"(random, singleton, enformer_low)",random,singleton,enformer_random
65537,chr10,79586224,SFTPA1|ENSG00000122852.15|EH38E2912112|10-7958...,G,T,1,PASS,AF=6.56953e-06;AC=1,10-79586224-G-T,1.358134,126_DNASE:LNCaP clone FG,"(random, singleton, enformer_low)",random,singleton,enformer_random
65561,chr2,210516723,CPS1|ENSG00000021826.18|EH38E2071165|2-2105167...,G,T,1,PASS,AF=6.58389e-06;AC=1,2-210516723-G-T,1.279766,601_DNASE:large intestin,"(random, singleton, enformer_low)",random,singleton,enformer_random


In [41]:
# investigate class distribution (variant_type) of new random set: same singleton and ultra-rare
print('new random set variant_type distribution')
print(new_random_classes['variant_type'].value_counts())

new random set variant_type distribution
variant_type
ultra-rare    2728
singleton     2728
Name: count, dtype: int64


In [42]:
# merge to tested results
bcMPRAlm_random = new_random_classes.merge(tested_bcMPRAlm_df, on='chr_pos_ref_alt', how='left')
bcMPRAlm_random # 5456
print('Number of bcMPRAlm results from random enformer class: ', bcMPRAlm_random.shape[0] )
print('Number of bcMPRAlm unique variants: ', bcMPRAlm_random['chr_pos_ref_alt'].nunique() )
bcMPRAlm_random_matchable = bcMPRAlm_random.loc[~bcMPRAlm_random['adj.P.Val'].isna()]
print('Number of bcMPRAlm random matchable: ',  bcMPRAlm_random_matchable.shape[0])

random_high, random_low = investigate_random_df(bcMPRAlm_random, high_enformer_threshold_df=min_dnase, low_enformer_threshold_df=max_dnase)

Number of bcMPRAlm results from random enformer class:  5456
Number of bcMPRAlm unique variants:  5456
Number of bcMPRAlm random matchable:  2857
Number of enformer_random rows with DNase_max above the min_dnase thresholds: 168
Number of enformer_random rows with DNase_max below the max_dnase thresholds: 38


I am having only 2857 randoms merged with MPRAlm results

In [43]:
# add neuros specific median to it



##### Investigate not matchable:
- more ultra-rare variants cannot be matched

In [44]:
# only not-matchable
not_matchable_bcMPRAlm_random = bcMPRAlm_random.loc[bcMPRAlm_random['adj.P.Val'].isna()]
not_matchable_bcMPRAlm_random[['gene_set', 'variant_type', 'enformer_class', 'ID']].groupby(['gene_set', 'variant_type', 'enformer_class']).count()

ID
gene_set variant_type enformer_class      
cardiac  singleton    enformer_random  340
         ultra-rare   enformer_random  368
cava     singleton    enformer_random  388
         ultra-rare   enformer_random  388
neuro    singleton    enformer_random  372
         ultra-rare   enformer_random  379
random   singleton    enformer_random  180
         ultra-rare   enformer_random  184

#### Analysis of presision and recall for random:
- enformer high:
  - from 5456 random sampled from the enformer dataset 
  - only 2859 have results in MPRAlm 
    - 101 above enformer threshold
    - 20 below enformer threshold
  - Significant result in random: 34
    - 3 above enformer threshold
    - 0 below enformer threshold
  - Recall/Sensitivity:  8.82%
  - Precision:  2.97%
  - Can we improve this by only looking on neuro specific predictions
- enformer low: 

In [45]:
# compute sensitivity and specificity 
# merge new random class to the mpralm results
new_random_classes
# # check duplicates no duplicates
# new_random_classes.loc[new_random_classes.duplicated(['CHROM', 'POS','REF','ALT'], keep=False)].sort_values(by='POS')

# merge with chr_pos_ref_alt
new_random_classes['chr_pos_ref_alt'] = new_random_classes['ID'].apply(hf.get_chrom_pos_ref_alt_pattern)
print(f'Number of random classes: {new_random_classes.shape[0]}')
# read the variant table 
bcMPRAlm_df = pd.read_csv(config['files']['creating']['toptable_bcMPRAlm_resequencing'], sep="\t")
bcMPRAlm_df
# only tested: 
tested_bcMPRAlm_df = bcMPRAlm_df.loc[bcMPRAlm_df['variant_id'].str.startswith('cardiac_neuro_cava_random')]
tested_bcMPRAlm_df
tested_bcMPRAlm_df['chr_pos_ref_alt'] = tested_bcMPRAlm_df['variant_id'].apply(hf.get_chrom_pos_ref_alt_pattern) # result is unique 
tested_bcMPRAlm_df.head() # logFC	AveExpr	t	P.Value	adj.P.Val	B	variant_id	chr_pos_ref_alt
mpralm_results_random = new_random_classes.merge(tested_bcMPRAlm_df, on='chr_pos_ref_alt', how='left')

merged_mpralm_results_random = mpralm_results_random.loc[~mpralm_results_random['adj.P.Val'].isna()]
not_merged_mpralm_results_random = mpralm_results_random.loc[mpralm_results_random['adj.P.Val'].isna()]
print(f'Number of merged_mpralm for random', merged_mpralm_results_random.shape[0]) # 2859, 27.05: 2857

Number of random classes: 5456
Number of merged_mpralm for random 2857


/tmp/ipykernel_55402/3827834329.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  tested_bcMPRAlm_df['chr_pos_ref_alt'] = tested_bcMPRAlm_df['variant_id'].apply(hf.get_chrom_pos_ref_alt_pattern) # result is unique


In [46]:
merged_mpralm_results_random.columns

Index(['CHROM', 'POS', 'ID', 'REF', 'ALT', 'QUAL', 'FILTER', 'INFO',
       'chr_pos_ref_alt', 'DNase_max', 'max_col', 'enformer_variant_info',
       'gene_set', 'variant_type', 'enformer_class', 'logFC', 'AveExpr', 't',
       'P.Value', 'adj.P.Val', 'B', 'variant_id'],
      dtype='object')

In [47]:
# check if mutliple matching on chr_pos_ref_alt 
# mpralm_results_random
merged_mpralm_results_random.duplicated(subset=['chr_pos_ref_alt']).sum()

0

In [56]:
# not_merged_mpralm_results_random = mpralm_results_random.loc[mpralm_results_random['adj.P.Val'].isna()]
# not_merged_mpralm_results_random.ID.to_list()

##### Enformer high

In [59]:
merged_mpralm_results_random.columns

Index(['CHROM', 'POS', 'ID', 'REF', 'ALT', 'QUAL', 'FILTER', 'INFO',
       'chr_pos_ref_alt', 'DNase_max', 'max_col', 'enformer_variant_info',
       'gene_set', 'variant_type', 'enformer_class', 'logFC', 'AveExpr', 't',
       'P.Value', 'adj.P.Val', 'B', 'variant_id'],
      dtype='object')

In [58]:
merged_mpralm_results_random[['enformer_variant_info', 'gene_set', 'variant_type']]

,enformer_variant_info,gene_set,variant_type
0,"(cardiac, ultra-rare, enformer_high)",cardiac,ultra-rare
1,"(cardiac, ultra-rare, enformer_high)",cardiac,ultra-rare
4,"(cardiac, ultra-rare, enformer_high)",cardiac,ultra-rare
5,"(cardiac, ultra-rare, enformer_high)",cardiac,ultra-rare
6,"(cardiac, ultra-rare, enformer_high)",cardiac,ultra-rare
...,...,...,...
5446,"(random, singleton, enformer_random)",random,singleton
5449,"(random, singleton, enformer_low)",random,singleton
5451,"(random, singleton, enformer_low)",random,singleton
5453,"(random, singleton, enformer_low)",random,singleton


In [49]:
total_number = merged_mpralm_results_random.shape[0]
all_high, all_low = investigate_random_df(merged_mpralm_results_random, high_enformer_threshold_df=min_dnase, low_enformer_threshold_df=max_dnase)
significant_merged_mpralm_results_random = merged_mpralm_results_random.loc[merged_mpralm_results_random['adj.P.Val'] < 0.05]
print(f'All significant in random: {significant_merged_mpralm_results_random.shape[0]}')
total_sig = significant_merged_mpralm_results_random.shape[0]
significant_high, significant_low = investigate_random_df(significant_merged_mpralm_results_random, high_enformer_threshold_df=min_dnase, low_enformer_threshold_df=max_dnase)
recall_enformer_high = significant_high / total_sig
precision_enformer_high = significant_high / all_high
print(f'Recall: ', recall_enformer_high)
print(f'Precision: ', precision_enformer_high)
# find numbers: 
# all high according to enformer (above enformer threshold for the group)
# all the significant from the subgroup
# all the significant from the high 
# # compute precision and recall
# recall_enformer = significant_high / total_sig
# recall_enformer
# precision_enformer = significant_high / all_high
# precision_enformer

Number of enformer_random rows with DNase_max above the min_dnase thresholds: 103
Number of enformer_random rows with DNase_max below the max_dnase thresholds: 16
All significant in random: 34
Number of enformer_random rows with DNase_max above the min_dnase thresholds: 2
Number of enformer_random rows with DNase_max below the max_dnase thresholds: 1
Recall:  0.058823529411764705
Precision:  0.019417475728155338


###### Sanity check of later created data

In [1]:
random_significant_table_path = "/home/kisa/coding/80K_MPRA/80K-Analysis/06_variant_analysis/notebooks/significant_random_class.tsv"
significant_random_class = pd.read_csv(random_significant_table_path, sep="\t")
significant_random_class['enformer_variant_info'] = significant_random_class['enformer_variant_info'].apply(literal_eval)

# Explode the list of tuples into separate rows
significant_random_class_exploded = significant_random_class.explode('enformer_variant_info')

# Split the tuples into separate columns
significant_random_class_exploded[['gene_set', 'variant_type', 'enformer_class']] = pd.DataFrame(significant_random_class_exploded['enformer_variant_info'].tolist(), index=significant_random_class_exploded.index)

# significant_random_class_exploded = significant_random_class.explode()
significant_high, significant_low = investigate_random_df(significant_random_class_exploded, high_enformer_threshold_df=min_dnase, low_enformer_threshold_df=max_dnase)


NameError: name 'pd' is not defined

With H1 dnase tracks: (done in /home/kisa/coding/80K_MPRA/80K-Analysis/06_variant_analysis/notebooks/investigate_H1_neuronal_stem_cell.ipynb)
- get the mpralm random results 
- check which can be matched with the screen data
- get number of these variants

##### Enformer low
- Measuring the precision and recall of the low classification:
  - sens/recall: true positive / all_true => low_class + non_significant / all_non_significant => 20 / (2859-34) = 0.71%
  - precision: true positive / all_positive_predicted => low_class + non_significant / low_class => 20 / 20 = 100%

In [55]:
total_number = merged_mpralm_results_random.shape[0]

all_high, all_low = investigate_random_df(merged_mpralm_results_random, high_enformer_threshold_df=min_dnase, low_enformer_threshold_df=max_dnase)
significant_merged_mpralm_results_random = merged_mpralm_results_random.loc[merged_mpralm_results_random['adj.P.Val'] < 0.05]
print(f'All significant in random: {significant_merged_mpralm_results_random.shape[0]}')
total_sig = significant_merged_mpralm_results_random.shape[0]
significant_high, significant_low = investigate_random_df(significant_merged_mpralm_results_random, high_enformer_threshold_df=min_dnase, low_enformer_threshold_df=max_dnase)
recall_enformer_low = (all_low - significant_low) / (total_number - total_sig)
precision_enformer_low = (all_low - significant_low) / all_low
print(f'Recall: ', recall_enformer_low)
print(f'Precision: ', precision_enformer_low)

# Number of enformer_random rows with DNase_max above the min_dnase thresholds: 101
# Number of enformer_random rows with DNase_max below the max_dnase thresholds: 20
# All significant in random: 34
# Number of enformer_random rows with DNase_max above the min_dnase thresholds: 3
# Number of enformer_random rows with DNase_max below the max_dnase thresholds: 0
# Recall:  0.007079646017699115
# Precision:  1.0

Number of enformer_random rows with DNase_max above the min_dnase thresholds: 103
Number of enformer_random rows with DNase_max below the max_dnase thresholds: 16
All significant in random: 34
Number of enformer_random rows with DNase_max above the min_dnase thresholds: 2
Number of enformer_random rows with DNase_max below the max_dnase thresholds: 1
Recall:  0.005313496280552604
Precision:  0.9375


In [52]:
significant_high = 2384 - 20 # non_significant_low
significant_not_high = 14094 + 2736 + 1 - 263 - 31 # non_significant not_low (all values not low - significant not low)
all_high = 2384 # all_low
all_not_high = 14094 + 2736 + 1 # all_not_low
all_values = all_high + all_not_high


total_sig = 14094 + 2736 + 2384 + 2 - 263 - 20 - 31 # total non-significant
total_non_sig = 263 + 31 + 20 # total significant
total_high = all_high 
total_non_high = all_not_high
table_total = 14094 + 2736 + 2384 + 2 # all

In [53]:

# compute resampling number:


for gene_set, variant_info in resampling_info.items():
    for variant_type, sampling_info in variant_info.items():
        print(f"Processing gene_set: {gene_set}, variant_type: {variant_type}")
        
        # 1. Get a DataFrame of the gene_set and variant_type
        subset_df = df[(df['gene_set'] == gene_set) & (df['variant_type'] == variant_type)]
        print(f"Initial subset size: {subset_df.shape[0]}")


# Apply the function to all gene_set and variant_type combinations
resampling_results = {}
for gene_set in total_number_dict.keys():
    resampling_results[gene_set] = {}
    for variant_type, _ in total_number_dict[gene_set]:
        resample_number_high, resample_number_low, random_removable_rows = calculate_resample_numbers(gene_set, variant_type)
        resampling_results[gene_set][variant_type] = {
            'resample_number_high': resample_number_high,
            'resample_number_low': resample_number_low,
            'random_removable_rows': random_removable_rows
        }


# compute numbers like 750 from 0.15 of total 


    
    
chosen_variants = prioritized_variant_number[gene_set][0][1]
total_number = total_number_dict[gene_set][0][1]

number_enformer_high = enformer_high_percentage * chosen_variants
number_enformer_low = enformer_low_percentage * chosen_variants
number_enformer_random = chosen_variants - (number_enformer_high + number_enformer_low)

for all_variants in [69363, 289724, 598537]:
    sampling_ration_random = 750/(all_variants - 750 - 3500)
    # sampling_ration_random = 750/(all_variants - 750 - 3500)
    # sampling_ration_random = 750/(all_variants - 750 - 3500)
    print(sampling_ration_random)
    random_sampling_num = round(sampling_ration_random * 4250)
    high_sampling_num = round(sampling_ration_random * 3500)
    low_sampling_num = random_sampling_num - high_sampling_num
    print(random_sampling_num, high_sampling_num, low_sampling_num)
    print(750 + high_sampling_num + low_sampling_num)
    print((750 + high_sampling_num + low_sampling_num)/all_variants)
    # 3500 * sampling_ration_random
    
    
# resampling 

resample_number_high = sampling_info['resample_number_high']
resample_number_low = sampling_info['resample_number_low']

if resample_number_high > 0:
    high_sample = subset_df[subset_df['enformer_class'] == 'enformer_high'].sample(n=resample_number_high, replace=False, random_state=1313)
    high_sample['enformer_class'] = 'enformer_random'
    new_rows.append(high_sample)
    print(f"Sampled {resample_number_high} from enformer_high to be added as enformer_random")
    
if resample_number_low > 0:
    low_sample = subset_df[subset_df['enformer_class'] == 'enformer_low'].sample(n=resample_number_low, replace=False, random_state=1313)
    low_sample['enformer_class'] = 'enformer_random'
    new_rows.append(low_sample)
    print(f"Sampled {resample_number_low} from enformer_low to be added as enformer_random")


NameError: name 'resampling_info' is not defined

#### Compute the fischer exact test and precision and recall
- merge the information to the design file and merge the design file and the mpra file + compute the significants => plots